# 04 - Classical ML

Random Forest baseline on the engineered pixel features from notebook 03, trained/evaluated on the real `sen1floods11` train/valid splits. Random Forest instead of the U-Net (also implemented, `src/ai/classic/unet.py`) here specifically because it trains on real per-pixel data in seconds on CPU - this notebook is meant to run end-to-end quickly and give a real number to compare the quantum kernel model (05) against, not to be the final production segmentation model.

In [1]:
import sys
sys.path.insert(0, r"d:\project-raw-data\sphoorthq-geoverse")

import numpy as np
from sklearn.ensemble import RandomForestClassifier

from src.ai.classic.sen1floods11_dataset import load_split, chip_id_from_s1_filename, read_s1, read_label
from src.fusion.pixel_features import build_feature_cube, cube_to_pixel_table
from src.ai.objectives.registry import evaluate
from src.observability.run_logger import RunLogger

logger = RunLogger("04_classical_ml")
RNG = np.random.default_rng(42)

## Build a pixel table from multiple chips

Uses a subsample of chips (not all 252 train / 89 valid) to keep this runnable in a couple of minutes on CPU - real data, real training, just bounded scope. Increase `N_TRAIN_CHIPS`/`N_VALID_CHIPS` for a fuller run.

In [2]:
N_TRAIN_CHIPS = 15
N_VALID_CHIPS = 8

def build_pixel_dataset(split: str, n_chips: int, seed: int = 0):
    pairs = load_split(split)
    rng = np.random.default_rng(seed)
    chosen = rng.choice(len(pairs), size=min(n_chips, len(pairs)), replace=False)

    xs, ys = [], []
    for idx in chosen:
        s1_filename, _ = pairs[idx]
        chip_id = chip_id_from_s1_filename(s1_filename)
        s1 = read_s1(chip_id)
        label = read_label(chip_id)
        cube = build_feature_cube(s1)
        x, y = cube_to_pixel_table(cube, label, raw_s1=s1)
        xs.append(x); ys.append(y)
    return np.concatenate(xs), np.concatenate(ys)

In [3]:
with logger.stage("build_train_pixel_table") as stage:
    x_train, y_train = build_pixel_dataset("train", N_TRAIN_CHIPS, seed=1)
    stage.metrics = {"n_chips": N_TRAIN_CHIPS, "n_pixels": int(len(y_train))}

with logger.stage("build_valid_pixel_table") as stage:
    x_valid, y_valid = build_pixel_dataset("valid", N_VALID_CHIPS, seed=2)
    stage.metrics = {"n_chips": N_VALID_CHIPS, "n_pixels": int(len(y_valid))}

print(f"train: {x_train.shape}, valid: {x_valid.shape}")

[04_classical_ml] -> build_train_pixel_table ...


[04_classical_ml] <- build_train_pixel_table [OK] 2.781s {'n_chips': 15, 'n_pixels': 3228826}
[04_classical_ml] -> build_valid_pixel_table ...


[04_classical_ml] <- build_valid_pixel_table [OK] 1.48s {'n_chips': 8, 'n_pixels': 1578750}
train: (3228826, 6), valid: (1578750, 6)


In [4]:
with logger.stage("train_random_forest") as stage:
    clf = RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
    clf.fit(x_train, y_train)
    stage.metrics = {"n_estimators": 200}

with logger.stage("evaluate_random_forest") as stage:
    y_pred = clf.predict(x_valid)
    metrics = evaluate("flood-segmentation", y_pred, y_valid)
    stage.metrics = {k: round(v, 4) for k, v in metrics.items()}

print("Random Forest validation metrics (real, on held-out sen1floods11 chips):")
for k, v in metrics.items():
    print(f"  {k:12s} {v:.4f}")

[04_classical_ml] -> train_random_forest ...


[04_classical_ml] <- train_random_forest [OK] 924.654s {'n_estimators': 200}
[04_classical_ml] -> evaluate_random_forest ...


[04_classical_ml] <- evaluate_random_forest [OK] 9.331s {'iou': 0.3582, 'f1': 0.5275, 'precision': 0.8384, 'recall': 0.3848, 'boundary_f1': 0.2365}
Random Forest validation metrics (real, on held-out sen1floods11 chips):
  iou          0.3582
  f1           0.5275
  precision    0.8384
  recall       0.3848
  boundary_f1  0.2365


In [5]:
import pickle

MODEL_PATH = r"d:\project-raw-data\sphoorthq-geoverse\datasets\processed\models\classical_rf_v1.pkl"
import os
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
with open(MODEL_PATH, "wb") as f:
    pickle.dump(clf, f)

logger.log_metrics({"model_path": MODEL_PATH, **{f"valid_{k}": v for k, v in metrics.items()}})
logger.finalize()
print(f"saved model -> {MODEL_PATH}")

[04_classical_ml] run complete in 938.621s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\e37afa6b-f0ca-43d5-af08-08f611813c2d.json
saved model -> d:\project-raw-data\sphoorthq-geoverse\datasets\processed\models\classical_rf_v1.pkl
